# 6주차 — LangSmith 추적과 디버깅 (Colab판)

「최신인공지능」 2026 · 6주차 실습

| 실습 | 교시 | 내용 |
|------|------|------|
| 사전 점검 | — | 네트워크·키 확인 |
| 실습 1 ★★ | 1교시 | **코드 수정 0줄** — 5주차 체인을 그대로 실행 |
| 2-5절 ★ | 1교시 | 추적 **끄는 법** (보안) |
| 3절 ★ | 2교시 | 병목 찾기 — 부모 Run 의 함정 |
| 실습 3 ★ | 3교시 | `@traceable` — 내 함수도 트리에 올린다 |
| 1-3절 ★★ | 3교시 | 태그·메타데이터로 실험 구분 |
| 실습 4 | 3교시 | Prompt Hub |

> ### ✅ 이 차시는 Colab 이 실습실보다 안전합니다
>
> `.env` 파일에 키를 적고 실수로 커밋하는 사고가 **구조적으로 일어나지 않습니다.**
> 대신 Colab 좌측 🔑 **보안 비밀** 패널을 씁니다.
>
> | 실습실 (원본) | Colab (이 노트북) |
> |---|---|
> | `.env` + `load_dotenv()` | 🔑 보안 비밀 + `userdata.get()` |
> | `git check-ignore -v .env` | (해당 없음 — 저장소에 안 들어감) |
> | `os.getenv("LANGSMITH_API_KEY")` | **똑같습니다** ★ |
>
> ⚠️ 다만 **"키를 코드 밖에 두고 이름으로만 참조한다"** 는 원칙은 동일합니다.
> 노트북 셀에 키를 직접 붙여넣지 마십시오 — 노트북은 공유되면 그대로 노출됩니다.

## 0. 준비 — LangSmith 가입과 키 등록

### ① 가입 (수업 전에 미리 해 두십시오)

https://smith.langchain.com — **신용카드 등록 불필요**

| 항목 | 내용 |
|---|---|
| 무료 한도 | 1인 1계정 / 월 5,000 traces / 추적 보존 14일 |
| 초과 시 | 429로 거부될 뿐 **과금되지 않음** |
| 주의 | **학생 개인 신용카드 등록 절대 금지** |

### ② 키 발급

`Settings` → `API Keys` → `Create API Key` → 값을 복사 (`lsv2_pt_...` 로 시작)

### ③ Colab 에 등록 ★

좌측 사이드바 🔑 **보안 비밀** → **새 보안 비밀 추가** → 아래 3개를 등록하고
각각 **"노트북 액세스"** 토글을 **켭니다**.

| 이름 | 값 |
|---|---|
| `LANGSMITH_API_KEY` | 발급받은 `lsv2_pt_...` |
| `LANGSMITH_PROJECT` | `week06-tracing` |
| `LANGSMITH_TRACING` | `true` |

> `LANGSMITH_TRACING` 과 `LANGSMITH_PROJECT` 는 비밀이 아니므로
> 아래 부트스트랩 셀이 **없으면 자동으로 채워 줍니다.** 등록해야 하는 것은 사실상 **키 하나**입니다.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat"]
WEEK_PACKAGES = "langchain langchain-core langchain-ollama python-dotenv pydantic langsmith"
WEEK_SECRETS  = ["LANGSMITH_API_KEY"]

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")

# ── LangSmith 설정 ────────────────────────────────────────────
# 🔑 보안 비밀에 넣어 둔 값을 환경변수로 옮긴다.
#    ★ 여기가 실습실의 load_dotenv() 자리입니다. 하는 일이 정확히 같습니다.
for k in WEEK_SECRETS + ["LANGSMITH_PROJECT", "LANGSMITH_TRACING"]:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass

# 비밀이 아닌 두 값은 없으면 여기서 채운다
os.environ.setdefault("LANGSMITH_PROJECT", "week06-tracing")
os.environ.setdefault("LANGSMITH_TRACING", "true")

print()
print("[키]  LANGSMITH_API_KEY   " + ("✅ 설정됨" if os.getenv("LANGSMITH_API_KEY") else "❌ 없음 — 🔑 보안 비밀에 등록하세요"))
print(f"[설정] LANGSMITH_PROJECT  = {os.getenv('LANGSMITH_PROJECT')}")
print(f"[설정] LANGSMITH_TRACING  = {os.getenv('LANGSMITH_TRACING')}")

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'")
print("=" * 62)

## 사전 점검 A — 네트워크

키 문제인지 네트워크 문제인지를 **먼저 갈라 둡니다.**
표준 라이브러리만 쓰므로 설치가 필요 없습니다.

> 💡 **401/403 도 성공으로 셉니다.** 서버가 응답을 준 것이므로
> "방화벽을 통과했다"는 뜻입니다. 우리가 보려는 것은 인증이 아니라 **연결**입니다.

In [ ]:
import socket, ssl, urllib.error, urllib.request

TIMEOUT = 8

TARGETS = [
    ("LangSmith 웹",  "https://smith.langchain.com"),
    ("LangSmith API", "https://api.smith.langchain.com/info"),
    ("Ollama 로컬",   "http://localhost:11434/api/tags"),
]


def probe(url: str) -> str:
    """접속 결과를 사람이 읽는 한 줄로 돌려준다."""
    try:
        with urllib.request.urlopen(url, timeout=TIMEOUT) as resp:
            return f"[O] HTTP {resp.status}  — 연결됨"
    except urllib.error.HTTPError as e:
        # 서버가 응답을 준 것 → 네트워크는 뚫려 있다
        return f"[O] HTTP {e.code}  — 연결됨 (인증/권한 응답이지 차단이 아님) ★"
    except urllib.error.URLError as e:
        reason = e.reason
        if isinstance(reason, ssl.SSLError):
            return f"[X] SSL 오류 — 프록시가 가로채는 중일 수 있음 ({reason})"
        if isinstance(reason, socket.timeout):
            return f"[X] 시간 초과 ({TIMEOUT}초) — 차단 의심"
        return f"[X] 연결 실패 — {reason}"
    except Exception as e:
        return f"[X] 예상 밖 오류 — {type(e).__name__}: {e}"


print("=" * 64)
print("  6주차 사전 점검 — 접속 확인")
print("=" * 64)
for name, url in TARGETS:
    print(f"  {name:14s} {url}")
    print(f"  {'':14s} → {probe(url)}")
    print()

| 결과 | 의미 / 조치 |
|---|---|
| LangSmith 웹·API 둘 다 `[O]` | 실습 진행 가능 ★ |
| 둘 다 `[X]` | 네트워크 문제. Colab에서 이 경우는 드뭅니다 |
| 웹만 `[O]`, API 는 `[X]` | 브라우저는 되는데 코드가 못 나가는 상태 ⚠️ |
| `Ollama` 만 `[X]` | `ollama serve` 가 안 떠 있는 것. **추적과 무관** — 부트스트랩 셀을 다시 실행 |

## 실습 1 ★ 준비 — 키 진단

`trace_on` 을 돌렸는데 목록이 비어 있으면 원인이 셋 중 하나입니다.
**그 셋을 여기서 미리 갈라 둡니다.**

> ★ 값 자체는 절대 출력하지 않습니다.
> **오늘은 추적 화면을 캡처해 제출합니다.** 캡처에 키가 통째로 찍히는 사고가 실제로 납니다.

In [ ]:
import os

NEW_NAMES = ["LANGSMITH_TRACING", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
OLD_NAMES = ["LANGCHAIN_TRACING_V2", "LANGCHAIN_API_KEY", "LANGCHAIN_PROJECT"]
TRUTHY    = {"true", "1", "yes", "on"}


def mask(value: str) -> str:
    """키가 맞는지 눈으로만 확인할 수 있게 가린다."""
    if len(value) <= 8:
        return "*" * len(value)
    return value[:8] + "*" * (len(value) - 8)


def show(names: list[str], label: str) -> dict[str, str]:
    print(f"  [{label}]")
    found: dict[str, str] = {}
    for name in names:
        value = os.getenv(name)
        if not value:
            print(f"    [X] {name:22s} 없음")
            continue
        found[name] = value
        if "API_KEY" in name:
            # ❌ print(value)  ← 절대 금지
            print(f"    [O] {name:22s} 로드됨  ({mask(value)}, {len(value)}자)")
        else:
            print(f"    [O] {name:22s} = {value}")
    print()
    return found


print("=" * 64)
print("  6주차 실습 1 — LangSmith 연동 확인")
print("=" * 64)

found = show(NEW_NAMES, "현재 권장 이름")
found.update(show(OLD_NAMES, "예전 이름 (있어도 대개 동작합니다)"))

# ── 자주 나오는 실수 세 가지를 여기서 잡는다 ★ ──
print("-" * 64)
print("  점검")
print("-" * 64)

tracing = found.get("LANGSMITH_TRACING") or found.get("LANGCHAIN_TRACING_V2") or ""
key     = found.get("LANGSMITH_API_KEY") or found.get("LANGCHAIN_API_KEY") or ""
project = found.get("LANGSMITH_PROJECT") or found.get("LANGCHAIN_PROJECT") or ""

# ① 스위치
if tracing.strip().lower() in TRUTHY:
    print("    [O] 추적 스위치가 켜져 있습니다.")
else:
    print(f"    [X] 추적이 꺼져 있습니다 (값: {tracing!r})")

# ② 키 — 가장 흔한 사고가 따옴표·공백입니다
if not key:
    print("    [X] API 키가 없습니다. Settings → API Keys 에서 발급하십시오.")
elif key != key.strip():
    print("    [X] 키 앞뒤에 공백이 있습니다 ⚠️")
elif key[0] in "\"'" or key[-1] in "\"'":
    print("    [X] 키가 따옴표로 감싸여 있습니다 ⚠️  → 따옴표 없이 붙여넣습니다.")
elif not key.startswith("lsv2_"):
    print(f"    [!] 키가 lsv2_ 로 시작하지 않습니다 (현재: {key[:5]}...)  🔶")
else:
    print("    [O] 키 형식이 정상으로 보입니다.")

# ③ 프로젝트
if project:
    print(f"    [O] 기록은 '{project}' 프로젝트로 들어갑니다.")
else:
    print("    [!] LANGSMITH_PROJECT 가 없습니다 → 'default' 로 들어갑니다.")

# ── 실제로 인증까지 되는지 확인 ──
print()
print("-" * 64)
print("  서버 확인")
print("-" * 64)
try:
    from langsmith import Client
    client = Client()
    list(client.list_projects(limit=1))     # 1건만 읽어 본다 — 인증되면 예외가 안 난다
    print("    [O] 인증 성공 — 내 계정으로 연결되었습니다 ★")
except ImportError:
    print("    [X] langsmith 패키지가 없습니다 → 부트스트랩 셀을 다시 실행")
except Exception as e:
    print(f"    [X] 연결/인증 실패 — {type(e).__name__}: {e}")
    print("        · 401/403 → 키가 잘못됨")
    print("        · 시간 초과 → 위 네트워크 점검부터 확인")

## 실습 1 ★★ (1교시) — 코드 수정 0줄

아래 셀에는 **추적 코드가 한 줄도 없습니다.**
LangSmith 를 import 하지도 않았고, 콜백을 붙이지도 않았습니다.

**바뀐 것은 환경변수 3개뿐입니다.** ★

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_PROJECT=week06-tracing
```

아래 체인은 **5주차 Least-to-Most 를 그대로 옮긴 것**입니다. 한 글자도 고치지 않았습니다.

| 2주차에 본 문제 | 추적이 해결하는 방식 |
|---|---|
| ① 비결정성 | 실행되는 모든 것이 자동으로 **기록**된다 |
| ② 다단계 | 중첩 실행이 **트리**로 펼쳐진다 |
| ③ 숨은 프롬프트 | 모델에 실제로 간 **최종 문자열**이 보인다 |

In [ ]:
import os
from typing import List

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

MODEL = os.environ["MODEL"]
TEMP  = 0

PROBLEM = (
    "한 카페가 오후 시간대 매출만 계속 줄고 있다. "
    "원인을 진단하고 개선안을 우선순위와 함께 제시하라."
)

llm = ChatOllama(model=MODEL, temperature=TEMP)


# ── ① 분해기 ────────────────────────────────────────────────────
class SubQuestions(BaseModel):
    """원 문제를 풀기 위해 순서대로 답해야 할 하위 질문들."""

    questions: List[str] = Field(description="쉬운 것부터 어려운 순서로 3~4개")


decompose_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문제를 잘게 쪼개는 조교다. 문제를 직접 풀지 마라."),
        ("human",  "다음 문제를 풀기 위해 순서대로 답해야 할 하위 질문으로 나눠라.\n\n{problem}"),
    ]
)
decomposer = decompose_prompt | llm.with_structured_output(SubQuestions)


# ── ② 풀이기 ────────────────────────────────────────────────────
solve_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문제 해결 조교다. 앞서 푼 결과를 근거로 이번 질문에만 짧게 답하라."),
        ("human",  "원 문제: {problem}\n\n지금까지 푼 것:\n{solved}\n\n이번 질문: {question}"),
    ]
)
solver = solve_prompt | llm | StrOutputParser()

# ★★ 2교시에서 이 prompt 의 Run 을 열어 봅니다.
#    코드에는 "{solved}" 라고만 적혀 있지만, 세 번째 하위 질문을 풀 때
#    모델이 실제로 받은 것은 1,000자가 넘는 실물 문자열입니다.
#    그게 1교시에서 말한 '숨은 프롬프트' 이고, 오늘 해결할 문제입니다.


# ── ③ 순서대로 푸는 반복문 ──────────────────────────────────────
def solve_in_order(data: dict) -> dict:
    problem = data["problem"]
    solved: List[str] = []

    for i, q in enumerate(data["subs"].questions, 1):
        answer = solver.invoke(
            {
                "problem":  problem,
                "solved":   "\n\n".join(solved) or "(아직 없음)",   # ← 누적된다 ★
                "question": q,
            }
        )
        solved.append(f"Q{i}. {q}\nA{i}. {answer}")
        print("─" * 60)
        print(solved[-1])

    return {"problem": problem, "solved": "\n\n".join(solved)}


# ── ④ 종합기 ────────────────────────────────────────────────────
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 최종 답변자다. 하위 풀이만을 근거로 답하라."),
        ("human",  "원 문제: {problem}\n\n하위 풀이:\n{solved}\n\n최종 답을 정리해라."),
    ]
)

# ── ⑤ 전부 파이프로 잇는다 ──────────────────────────────────────
chain = (
    RunnablePassthrough.assign(subs=decomposer)
    | RunnableLambda(solve_in_order)
    | final_prompt
    | llm
    | StrOutputParser()
)

print("=" * 60)
print("원 문제:", PROBLEM)
print("=" * 60)

answer = chain.invoke({"problem": PROBLEM})       # ← 1회 = 1 trace ★

print("=" * 60)
print("[최종 답]")
print(answer)
print("=" * 60)

### 이제 브라우저로 갑니다 ★

`smith.langchain.com` → 좌측 **Projects** → **week06-tracing**
→ 방금 실행이 목록에 한 줄로 찍혀 있습니다.

> 🎯 **오늘 1교시의 목표는 이 장면 하나입니다.**
> "내가 아무것도 안 했는데 다 기록돼 있다."
> 코드는 5주차 그대로이고, 추가한 것은 환경변수 3개뿐입니다.
>
> ⚠️ 트리를 펼쳐 보는 것은 2교시입니다. 여기서는 **"기록됐다"** 까지만!

**목록에 아무것도 없다면**

| 증상 | 원인 | 조치 |
|---|---|---|
| 목록이 비어 있음 | `LANGSMITH_TRACING ≠ true` | 위 진단 셀 재실행 |
| 인증 오류 | 키 앞뒤 공백·따옴표 | 🔑 보안 비밀에서 따옴표 제거 |
| 값을 고쳤는데 안 바뀜 ★ | 커널에 옛 값이 남음 | **[런타임] > [세션 다시 시작]** 후 처음부터 |
| 접속 자체가 안 됨 | 네트워크 | 위 `probe` 셀 확인 |

## 1교시 2-5절 ★ — 끄는 법 (반드시 확인하십시오)

> ### ⚠️⚠️ 추적을 켜면 프롬프트와 응답이 외부 서비스로 전송됩니다
>
> | 넣어도 되는 것 | 절대 넣으면 안 되는 것 |
> |---|---|
> | 수업용 예제 문장 | 주민번호·연락처 등 개인정보 |
> | 공개 문서 | 회사 기밀·미공개 자료 |
> | 직접 만든 테스트 데이터 | 실제 고객 데이터 |
>
> **4주차에 배운 보안 기준을 기억하십니까?** 로컬 모델을 쓰면 데이터가 안 나간다고 했었죠.
> **추적을 켜면 그 이점이 사라집니다.**

아래 셀은 같은 질문을 **3번** 던집니다.
그런데 LangSmith 목록에는 **2건만** 찍혀야 정상입니다. ★

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 한 문장으로만 답하는 조교다."),
        ("human",  "{question}"),
    ]
)
chain = prompt | ChatOllama(model=os.environ["MODEL"], temperature=0) | StrOutputParser()
Q = "LangSmith 는 무엇을 하는 도구인가?"

print("=" * 64)
print(f"  현재 LANGSMITH_TRACING = {os.getenv('LANGSMITH_TRACING')!r}")
print("=" * 64)

# ── ① 그냥 실행 — 기록된다 ──
print("[1] 기본 실행 — 이건 기록됩니다")
print("   ", chain.invoke({"question": Q}))
print()

# ── ② 구간만 끈다 ★ — 민감한 데이터를 다루는 그 구간만 감싸면 됩니다 ──
print("[2] tracing_context(enabled=False) 로 감싼 실행 — 기록되지 않습니다 ★")
try:
    from langsmith import tracing_context
except ImportError:
    # 🔶 langsmith 버전에 따라 위치가 다릅니다.
    from langsmith.run_helpers import tracing_context

with tracing_context(enabled=False):
    print("   ", chain.invoke({"question": Q + " (이 실행은 기록되지 않아야 한다)"}))
print()

# ── ③ 다시 켜진다 ──
print("[3] with 블록을 나오면 원래대로 — 이건 다시 기록됩니다")
print("   ", chain.invoke({"question": Q}))

### 확인 ★

`smith.langchain.com` → `week06-tracing` 프로젝트를 **새로고침**합니다.
방금 **3번** 호출했는데 목록에는 **2건만** 늘어나 있어야 합니다. `[2]` 번이 빠진 것입니다.

### 세 가지 끄는 방법

| 범위 | 방법 | 언제 쓰나 |
|---|---|---|
| 전체 | `LANGSMITH_TRACING=false` (Colab은 세션 재시작 필요 ★) | 대량 실행 · 한도 절약 |
| 구간 | `with tracing_context(enabled=False):` | 민감 데이터 구간만 |
| 프로젝트 | `LANGSMITH_PROJECT` 를 바꿔 격리 | 섞이는 것만 막고 싶을 때 |

> ⚠️ 강의안에 적힌 `chain.invoke(x, config={"callbacks": []})` 는
> 버전에 따라 **기대대로 꺼지지 않습니다.** 환경변수로 추적이 켜져 있으면
> 프레임워크가 추적기를 다시 붙이기 때문입니다.
> → 확실히 끄려면 위의 `tracing_context` 나 환경변수를 쓰십시오. ★
>
> 💡 **실무 습관**: 개발할 때 켜고, 대량 실행할 때 끕니다.

## 2교시 3절 ★ — 병목 찾기: 부모 Run 의 함정

세 단계 중 **② 만 일부러 길게** 답하게 만들어 두었습니다.
추적 화면에서 ② 가 압도적으로 오래 걸린 것이 한눈에 보여야 합니다.

```
Run: 진단 파이프라인            (부모 — 항상 1등이다. 병목이 아니다 ⚠️)
 ├─ ① 요약      짧게       ~2초
 ├─ ② 상세 분석  아주 길게  ~12초   ★ 여기가 진짜 병목
 └─ ③ 한 줄 결론 짧게       ~2초
```

### ★★ 부모 Run 의 함정

```
전체 시간 = 자기 + 자식 전부
자기 시간 = 전체 시간 − 자식들의 시간   ← 병목은 이것이 큰 Run ★
```

> "부장님이 제일 오래 걸렸다고 하면 안 되죠. 팀 전체 시간이니까요.
> **누가 실제로 오래 잡고 있었는지**를 봐야 합니다."

**진단 4단계**

1. 트리를 펼친다
2. **잎**(자식이 없는 Run)만 본다 ← 여기서 자기 시간 = 전체 시간 ★
3. 가장 큰 것을 고른다
4. 그 Run 의 입력·출력·토큰을 열어 '왜 오래 걸렸는지' 추정한다

In [ ]:
import os, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama

MODEL = os.environ["MODEL"]
TOPIC = "한 카페의 오후 시간대 매출 감소"

llm = ChatOllama(model=MODEL, temperature=0)


def step(system: str, human: str):
    return (ChatPromptTemplate.from_messages([("system", system), ("human", human)])
            | llm | StrOutputParser())


# ── ① 짧게 ──────────────────────────────────────────────────────
summarize = step(
    "너는 요약 담당이다. 반드시 두 문장 이내로만 답한다.",
    "다음 상황을 요약해라: {topic}",
)

# ── ② 일부러 길게 ★ — 출력 토큰이 많으면 느려진다 ──────────────
analyze = step(
    "너는 분석 담당이다. 아주 상세하게, 최소 900자 이상으로 답한다. "
    "가능한 모든 원인을 빠짐없이 나열하고 각각을 길게 설명한다.",
    "다음 요약을 근거로 원인을 낱낱이 분석해라:\n{summary}",
)

# ── ③ 짧게 ──────────────────────────────────────────────────────
conclude = step(
    "너는 결론 담당이다. 반드시 한 문장으로만 답한다.",
    "다음 분석의 결론을 한 문장으로:\n{analysis}",
)

# ── 로컬에서도 단계별 시간을 재 둔다 ────────────────────────────
#    추적 화면의 값과 대조시키기 위한 것입니다.
#    (4주차에서 VRAM 계산값과 ollama ps 실측을 대조한 것과 같은 방식)
TIMES: dict[str, float] = {}


def timed(name: str, runnable):
    """단계 하나를 감싸 실행 시간을 재고, 트리에 표시될 이름도 지정한다."""

    def _run(data: dict) -> str:
        t0 = time.perf_counter()
        out = runnable.invoke(data)
        TIMES[name] = time.perf_counter() - t0
        return out

    return RunnableLambda(_run).with_config(run_name=name)


step1 = timed("① 요약",      summarize)
step2 = timed("② 상세 분석", analyze)
step3 = timed("③ 한 줄 결론", conclude)


def pipeline(data: dict) -> dict:
    """세 단계를 순서대로 돈다 — 이 함수가 '부모 Run' 이 된다."""
    summary  = step1.invoke({"topic": data["topic"]})
    analysis = step2.invoke({"summary": summary})
    verdict  = step3.invoke({"analysis": analysis})
    return {"summary": summary, "analysis": analysis, "verdict": verdict}


slow = RunnableLambda(pipeline).with_config(run_name="진단 파이프라인")


def report(total: float) -> None:
    child = sum(TIMES.values())
    self_time = total - child
    print()
    print("  단계            시간(초)    비율     비고")
    print("  " + "─" * 62)
    print(f"  진단 파이프라인 {total:8.2f}   {100:5.1f}%   ← 부모. 항상 1등이다 (병목 아님) ⚠️")
    for name, sec in TIMES.items():
        mark = "  ★ 병목" if sec == max(TIMES.values()) else ""
        print(f"    {name:12s} {sec:8.2f}   {sec / total * 100:5.1f}%{mark}")
    print(f"    (부모 자기시간){self_time:8.2f}   {self_time / total * 100:5.1f}%   ← 부모가 '실제로' 쓴 시간 ★")
    print("  " + "─" * 62)


def run_pipeline_once(i: int, n: int) -> None:
    TIMES.clear()
    print("=" * 66)
    print(f"  실행 {i}/{n} — 주제: {TOPIC}")
    print("=" * 66)
    t0 = time.perf_counter()
    result = slow.invoke({"topic": TOPIC})
    total = time.perf_counter() - t0
    print(f"  [결론] {result['verdict']}")
    report(total)
    print()


run_pipeline_once(1, 1)

In [ ]:
# ★ 두 번 돌려 '워밍업 효과' 를 비교합니다.
#   첫 실행이 유독 느렸다면 그건 모델 문제가 아니라 워밍업입니다.
#   추적 화면에서도 두 Trace 를 나란히 놓고 보십시오. ★
for i in (1, 2):
    run_pipeline_once(i, 2)

### 추적 화면에서 확인할 것 ★

| 관찰 | 읽는 법 |
|---|---|
| '진단 파이프라인' 이 제일 김 | 자식 전부를 감쌌으니 당연하다. **병목 아님** ⚠️ |
| ② 상세 분석 이 제일 김 | **잎 Run 중 1등 = 진짜 병목** ★ |
| ② 의 출력 토큰이 크다 | 느린 원인 = '답을 길게 쓰고 있음' → 처방: `max_tokens` · 프롬프트로 길이 제한 |

### 느린 원인 세 가지와 처방

| 관찰 | 원인 | 처방 | 배운 곳 |
|---|---|---|---|
| 출력 토큰이 많다 | 답이 길다 | `max_tokens` / 길이 제한 | 4주차 3교시 |
| 입력 토큰이 많다 | 누적된 컨텍스트 | 앞 결과를 요약해 넘김 | 5주차 3교시 |
| **토큰은 적은데 느리다** ★ | 모델 로딩·CPU 분산 | `ollama ps` 의 `PROCESSOR` | 4주차 1교시 ★★ |

> ⚠️ **지연 ≠ 비용.** 토큰으로 정렬하면 1등이 달라질 수 있습니다.
> "어디를 고칠 것인가" 는 "무엇을 개선하려는가" 에 따라 답이 달라집니다.

## 실습 3 ★ (3교시) — `@traceable`: 내 함수도 트리에 올린다

2교시에서 본 트리에는 LangChain 부품만 있었습니다. **내 코드는 안 보였습니다.**

```python
raw    = open("review.txt").read()      # ← 트리에 안 나옴
text   = preprocess(raw)                # ← 트리에 안 나옴  ★
result = chain.invoke({"review": text}) # ← 이것만 나옴
save(result)                            # ← 트리에 안 나옴
```

**핵심 질문**: 전처리에서 텍스트를 잘못 잘라 먹었다면, 추적 화면으로 알 수 있습니까?

| 상황 | 추적에 보이는가 |
|---|---|
| 체인이 이상한 답을 냄 | ✅ |
| 체인에 들어간 **입력이 이미 망가져 있었음** | ❌ 안 보임 ★ |
| 전처리가 5초 걸림 | ❌ 안 보임 |

> ⚠️ **이게 실무에서 가장 흔한 오진입니다.**
> "모델이 이상하다"고 몇 시간을 보고, 알고 보면 전처리에서 문서 절반이 날아가 있던 경우입니다.
> **10주차 RAG 실습에서 반드시 겪게 됩니다.**

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langsmith import traceable          # ★ langchain 이 아니라 langsmith

MODEL = os.environ["MODEL"]
LIMIT = 500                              # ★ 아래에서 50 으로 바꿔 '전처리 버그' 를 만듭니다

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 고객 리뷰 분석기다. 장점과 단점을 각각 한 줄로 정리한다."),
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)
review_chain = prompt | ChatOllama(model=MODEL, temperature=0) | StrOutputParser()


# ── @traceable — 데코레이터 한 줄 ★ ─────────────────────────────
@traceable(name="리뷰 전처리", run_type="tool")
def preprocess(raw: str) -> str:
    """공백·개행을 정리하고 정해진 길이로 자른다."""
    return raw.replace("\n", " ").strip()[:LIMIT]


@traceable(name="결과 후처리", run_type="tool")
def postprocess(text: str) -> dict:
    return {"length": len(text), "text": text}


@traceable(name="리뷰 분석 파이프라인")       # ★★ 전체를 감싼다
def run_pipeline(raw: str) -> dict:
    text   = preprocess(raw)                    # ← 자식 Run ①
    result = review_chain.invoke({"review": text})  # ← 자식 Run ② (체인 트리 전체)
    return postprocess(result)                  # ← 자식 Run ③


# ★ run_pipeline 으로 전체를 감싼 것이 요령입니다.
#   감싸지 않으면 전처리·체인·후처리가 각각 별개의 Trace 로 흩어집니다.
#
#   인자          하는 일
#   ─────────────────────────────────────────────────────────
#   name=         트리에 표시될 이름 (없으면 함수 이름)
#   run_type=     Run 종류 — "tool" · "chain" · "retriever" 등
#   (인자·반환값)  자동으로 Input / Output 에 기록됨 ★
#
# ⚠️ 함수의 인자와 반환값이 그대로 기록됩니다.
#    개인정보가 인자로 들어가면 그대로 외부로 전송됩니다.

RAW = """  이 무선 이어폰 배터리는
정말 오래갑니다. 한 번 충전하면 사흘은 쓰네요.
음질도 이 가격대에서는 훌륭한 편입니다.
다만 케이스가 좀 크네요. 주머니에 넣으면 불룩합니다.
그리고 통화 품질은 기대 이하였습니다. 상대방이 잘 안 들린다고 합니다.  """

print("=" * 66)
print(f"  @traceable 실습 — 전처리 길이 제한 LIMIT = {LIMIT}자")
print("=" * 66)
out = run_pipeline(RAW)
print(f"  원문 길이     : {len(RAW)}자")
print("-" * 66)
print(out["text"])
print("=" * 66)

In [ ]:
# ★ 이번에는 LIMIT 을 50 으로 낮춰 '전처리 버그' 를 만들어 봅니다.
LIMIT = 50

print("=" * 66)
print(f"  @traceable 실습 — 전처리 길이 제한 LIMIT = {LIMIT}자")
print("=" * 66)
out = run_pipeline(RAW)
print("-" * 66)
print(out["text"])
print("=" * 66)

print("""
⚠️ 방금 무슨 일이 일어났습니까?

   체인의 출력이 엉뚱해졌습니다. 리뷰의 뒷부분(케이스·통화 품질)이
   아예 모델에 가지 않았기 때문입니다.

   ★ 그런데 그 원인이 추적 화면의 '리뷰 전처리' Run 의 Output 에
     그대로 보입니다. 50자짜리 잘린 문자열이 실물로 찍혀 있습니다.

   "모델이 이상한 게 아니었다."
   이 장면을 한 번 겪으면 이 절이 완성됩니다. ★
""")

### 추적 화면에서 이렇게 보입니다 ★

```
Run: 리뷰 분석 파이프라인            ← @traceable (내 함수)
 ├─ Run: 리뷰 전처리                 ← @traceable  ★ 새로 보임
 ├─ Run: RunnableSequence            ← LangChain 체인 (2교시에 본 것)
 │    ├─ Run: ChatPromptTemplate
 │    └─ Run: ChatOllama
 └─ Run: 결과 후처리                 ← @traceable  ★ 새로 보임
```

| 관찰 항목 | 짚어줄 말 |
|---|---|
| 내 함수가 트리에 나타남 | "이제 전처리 버그도 여기서 잡습니다" |
| 전처리 Run 의 Output | **잘린 결과가 실물로 보인다** ★ |
| 파이프라인이 하나의 Trace | 감싸지 않았으면 3개로 흩어졌을 것 |

## 3교시 1-3절 ★★ — 태그·메타데이터로 실험을 구분한다

**핵심 질문**: 프롬프트를 고쳐 다시 돌렸습니다. 목록에 실행이 20개 쌓여 있습니다.
**어느 게 고치기 전이고 어느 게 고친 후입니까?**

```
Traces 목록
├─ 14:32:10   ...    ← ???
├─ 14:33:02   ...    ← ???
├─ 14:35:41   ...    ← ???
     시각 말고는 구분할 방법이 없다 ⚠️
```

**해결**: 실행할 때 꼬리표를 붙입니다.

| | 태그(`tags`) | 메타데이터(`metadata`) |
|---|---|---|
| 형태 | 문자열 리스트 | 키-값 dict |
| 쓰임 | 목록에서 빠르게 필터 | 조건 검색·비교, 값 자체를 봄 |
| 예 | `["exp-A", "night-run"]` | `{"temperature": 0.8, "k": 5}` |

> ### ★★ 왜 지금 이걸 배우는가
>
> 다음 주 7주차의 주제가 **"바꾼 게 정말 좋아졌는가"** 입니다.
> 비교하려면 **비교 대상이 구분되어 있어야** 합니다.
> 태그 없이 100개를 쌓아두면 다음 주에 비교할 수가 없습니다.
>
> "실험은 하는 것보다 **구분해 두는 것**이 중요합니다.
> 라벨 없는 시험관 100개는 실험이 아닙니다."

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL  = os.environ["MODEL"]
REVIEW = ("이 무선 이어폰 배터리는 정말 오래갑니다. 다만 케이스가 좀 크고, "
          "통화 품질은 기대 이하였습니다.")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 고객 리뷰 분석기다. 장점과 단점을 각각 한 줄로 정리한다."),
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)


def build(temperature: float):
    return prompt | ChatOllama(model=MODEL, temperature=temperature) | StrOutputParser()


# ── 방법 A: 호출할 때 config 로 붙인다 ──────────────────────────
result = build(0.0).invoke(
    {"review": REVIEW},
    config={
        "tags": ["exp-A", "temp-0.0"],       # ★ 필터용 짧은 라벨
        "metadata": {                         # ★ 검색·비교용 키-값
            "variant": "A",
            "model": MODEL,
            "temperature": 0.0,
            "note": "프롬프트 수정 전",
        },
    },
)
print("[방법 A — config 로 붙이기]")
print(" ", result.replace("\n", " ")[:80], "...")

In [ ]:
# ── 방법 B: 체인에 미리 붙여 두고 파생한다 ★ ────────────────────
#    같은 체인에서 라벨만 다른 변형을 만들어 두는 방식입니다.
print("[방법 B — with_config 로 파생해 A/B 를 나란히 만든다] ★")

for variant, temp in [("A", 0.0), ("B", 0.8)]:
    c = build(temp).with_config(
        tags=[f"exp-{variant}", f"temp-{temp}"],
        metadata={
            "variant": variant,
            "model": MODEL,
            "temperature": temp,        # ★ 설정값을 함께 기록해야 '재현' 이 된다
            "run_by": "week06-practice",
        },
    )
    out = c.invoke({"review": REVIEW})
    print(f"  exp-{variant} (temp={temp}) → {out.replace(chr(10), ' ')[:70]} ...")

### 이제 웹에서 확인합니다 ★

`smith.langchain.com` → `week06-tracing` → 필터에 **`exp-A`** 를 입력

- `exp-A` 로 태그된 실행만 남습니다. `exp-B` 로 바꾸면 다른 쪽만 남습니다.
- Run 하나를 열어 `Metadata` 를 보면 `temperature` · `model` 이 그대로 있습니다.

> ★ **설정값을 함께 기록해 두지 않으면**, 나중에 "이건 몇 도로 돌린 거지?" 를
> 알 방법이 없습니다. **재현이 안 되는 실험은 실험이 아닙니다.**

## 실습 4 (3교시) — Prompt Hub

5주차에서 "프롬프트는 자산"이라고 했습니다. 지금은 이렇게 관리합니다.

```python
prompt = ChatPromptTemplate.from_messages([...])   # ← .py 파일 안에 박혀 있음
```

Git 으로 버전 관리는 됩니다. 그런데 문제가 남습니다.

| 문제 | 설명 |
|---|---|
| 프롬프트만 고쳐도 배포해야 함 | 코드 파일이므로 |
| **개발자가 아니면 못 고침** ★ | 기획자·도메인 전문가가 손을 못 댐 |
| 여러 프로젝트에서 복사됨 | 같은 프롬프트가 5군데에 흩어짐 |
| 어떤 버전이 돌고 있는지 불명확 | 코드를 뒤져야 함 |

```
[코드에 박아둘 때]              [Hub 로 뺄 때]
프롬프트 수정 → 배포 → 반영      프롬프트 수정 → 저장 → 반영 ★
```

> 🔶🔶 **이 셀은 이 차시에서 가장 깨지기 쉬운 코드입니다.**
> `push_prompt` / `pull_prompt` 의 인자 이름은 `langsmith` 버전에 따라 다릅니다.
> **수업 전날 반드시 1회 실행해 동작하는 형태로 확정하십시오.**
> 실패해도 아래 셀은 진단 메시지를 내고 넘어갑니다 — 수업이 멈추지 않습니다.

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langsmith import Client

MODEL = os.environ["MODEL"]
NAME  = "review-analyzer"        # Hub 에 올릴 이름

client = Client()

# ── ① 올릴 프롬프트 (5주차에 만든 것) ──────────────────────────
prompt_v1 = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 고객 리뷰 분석기다."),
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)

# ── ③ 고친 프롬프트 — 같은 이름으로 다시 push 하면 새 버전이 쌓인다 ──
prompt_v2 = ChatPromptTemplate.from_messages(
    [
        # ↓ 이 한 줄이 수정분입니다
        ("system", "너는 고객 리뷰 분석기다. 과장하지 말고 리뷰에 있는 사실만 사용하라."),
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)


def push(obj, label: str):
    """Hub 에 등록한다. 같은 이름이면 새 커밋(버전)이 쌓인다. ★"""
    try:
        url = client.push_prompt(NAME, object=obj)
        print(f"  [O] {label} 등록됨\n      {url}")
        return url
    except Exception as e:
        print(f"  [X] {label} 등록 실패 — {type(e).__name__}: {e}")
        print("      🔶 langsmith 버전에 따라 인자 이름이 다릅니다.")
        print("         help(client.push_prompt) 로 시그니처를 확인하십시오.")
        return None


print("=" * 66)
print("  실습 4 — Prompt Hub")
print("=" * 66)
print("\n[② 등록]")
push(prompt_v1, "v1")
print("\n[③ 고쳐서 다시 등록 → 새 버전이 생긴다]")
push(prompt_v2, "v2")

# ── 버전 이력 확인 — 웹의 Commits 탭과 같은 내용 ──
print("\n  버전 이력 (최근 5건) ★")
try:
    for c in client.list_prompt_commits(NAME, limit=5):
        print(f"    · {str(getattr(c, 'commit_hash', '?'))[:8]}")
except Exception as e:
    print(f"    [!] 커밋 목록을 못 읽었습니다 — {type(e).__name__}: {e}")
    print("        웹의 Prompts → 해당 프롬프트 → Commits 탭에서 확인하십시오.")

In [ ]:
# ── ④ 불러오기 — 불러온 것은 '평범한 ChatPromptTemplate 객체' 다 ★ ──
try:
    loaded = client.pull_prompt(NAME)                    # 최신 버전
    # loaded = client.pull_prompt(f"{NAME}:<커밋해시>")   # 특정 버전 고정 ★

    print(f"  [O] 불러온 객체 타입: {type(loaded).__name__}")

    # ★ 관찰 포인트: | 로 이어붙이는 방식이 하나도 안 바뀝니다.
    #   5주차에서 배운 Runnable 규약 덕분입니다.
    hub_chain = loaded | ChatOllama(model=MODEL, temperature=0) | StrOutputParser()
    out = hub_chain.invoke({"review": "배터리가 오래갑니다. 다만 케이스가 큽니다."})
    print("  실행 결과:", out.replace("\n", " ")[:80], "...")
except Exception as e:
    print(f"  [X] 불러오기 실패 — {type(e).__name__}: {e}")
    print("      🔶 소유자 접두사가 필요할 수 있습니다: '<내계정>/review-analyzer'")

### 웹에서 확인 — Prompts 메뉴

| 화면에서 볼 것 | 의미 |
|---|---|
| 프롬프트 본문 | 등록된 실물 |
| **Commits (버전 이력)** ★ | 고칠 때마다 쌓인다 |
| 공개/비공개 설정 | 기본은 비공개 |
| Playground | 웹에서 바로 시험 실행 ★ |

### 버전을 고정하는 법

| 방식 | 언제 쓰나 |
|---|---|
| 최신 버전 (`"이름"`) | 개발 중 — 항상 최신을 따라감 |
| **버전 고정** (`"이름:커밋"`) ★ | 운영 중 — 남이 고쳐도 내 서비스는 안 흔들림 |

> ⚠️ **운영 서비스는 반드시 버전을 고정해야 합니다.**
> 최신을 따라가게 두면 누군가 프롬프트를 고친 순간 내 서비스 동작이 바뀝니다.
> **배포한 적도 없는데요.** — `requirements.txt` 의 버전 고정과 같은 이유입니다.

### ⚖️ 대가도 있습니다

| 얻는 것 | 잃는 것 |
|---|---|
| 코드 배포 없이 수정 | 외부 서비스 의존 — 장애 시 못 가져옴 ⚠️ |
| 비개발자도 수정 가능 | 아무나 고치면 아무도 모르게 동작이 바뀜 |
| 버전 이력·롤백 | 코드 저장소와 이력이 두 군데로 갈림 |
| 웹에서 바로 시험 | **오프라인 실행 불가** ★ |

> ★ 4주차에서 "로컬 모델의 장점 = 오프라인 가능" 이라고 적었습니다.
> 그런데 프롬프트를 Hub 에서 가져오면 인터넷이 없으면 안 돕니다.
> **로컬 모델을 쓰는 의미가 반쯤 사라집니다.**
> 실무 절충안: 시작할 때 한 번 pull 해서 로컬에 캐시해 두고 씁니다.
>
> 📌 **본 교과목 방침**: 미니 프로젝트에 Prompt Hub 는 **필수가 아닙니다.**
> "이런 방식이 있고, 무엇을 주고받는지"를 아는 것까지가 오늘의 목표입니다.

## 오늘 확인할 것

- [ ] LangSmith 가입 + 키를 🔑 보안 비밀에 등록했다
- [ ] **코드 수정 0줄**로 5주차 체인이 기록되는 것을 봤다 ★★
- [ ] `tracing_context(enabled=False)` 로 **끄는 법**을 확인했다 (보안) ★
- [ ] 트리에서 **잎 Run 중 1등**을 골라 병목을 찾았다 ★
- [ ] `@traceable` 로 내 함수를 트리에 올렸다 ★
- [ ] `LIMIT=50` 버그를 **전처리 Run 의 Output** 에서 눈으로 잡았다 ★★
- [ ] 태그로 `exp-A` / `exp-B` 를 필터해 봤다 ★★

### 과제 2

5주차 **Least-to-Most 체인**을 추적한 화면을 캡처해 제출합니다.

> ⚠️ **캡처에 API 키가 찍히지 않도록 확인하십시오.** ★